# Formatação Final do Excel: Dicionários e Prova Real
Este notebook está organizado em módulos. Cada célula de código executa uma etapa específica do processo de formatação e exportação dos dados. As células em markdown explicam o propósito de cada módulo.

## 1. Carregando a base analítica
Carrega a base de dados analítica calculada previamente.

In [ ]:
import pandas as pd
import os

caminho_bd = '../banco_de_dados/'
caminho_csv = caminho_bd + 'Base_Analitica_IVS_Calculado.csv'
df_ivs = pd.read_csv(caminho_csv, sep=';', dtype={'CD_SETOR': str})

## 2. Criando o Dicionário de Dados
Gera um dicionário com as variáveis finais, tipos e descrições.

In [ ]:
dados_dic = [
    {'Variável Final': 'CD_SETOR', 'Tipo': 'Texto', 'Descrição': 'Código oficial do Setor Censitário (IBGE).'},
    {'Variável Final': 'NM_MUN', 'Tipo': 'Texto', 'Descrição': 'Nome do Município.'},
    {'Variável Final': 'NM_BAIRRO', 'Tipo': 'Texto', 'Descrição': 'Nome do Bairro.'},
    {'Variável Final': 'SITUACAO', 'Tipo': 'Texto', 'Descrição': 'Situação do setor (Ex: Urbana, Rural).'},
    {'Variável Final': 'ind_agua_inadequada', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proporção de domicílios sem acesso a rede geral de água.'},
    {'Variável Final': 'ind_esgoto_inadequado', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proporção de domicílios com esgotamento sanitário precário ou inexistente.'},
    {'Variável Final': 'ind_lixo_inadequado', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proporção de domicílios com destinação de lixo inadequada (queimado, enterrado, etc).'},
    {'Variável Final': 'ind_analfabetismo', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proporção de pessoas com 15 anos ou mais que não sabem ler e escrever.'},
    {'Variável Final': 'ind_cor_raca', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Proporção de pessoas autodeclaradas pretas, pardas ou indígenas (Vulnerabilidade Demográfica).'},
    {'Variável Final': 'ind_densidade_habitacional', 'Tipo': 'Numérico (> 0)', 'Descrição': 'Média exata de moradores por domicílio particular ocupado.'},
    {'Variável Final': 'ind_vulnerabilidade_renda', 'Tipo': 'Numérico (0 a 1)', 'Descrição': 'Índice normalizado de renda (Min-Max invertido). Mais próximo de 1 = Mais pobre.'},
]
df_dic_dados = pd.DataFrame(dados_dic)

## 3. Criando o Dicionário de Aplicação
Gera um dicionário com as fórmulas aplicadas e variáveis originais do Censo 2022.

In [ ]:
dados_app = [
    {'Dimensão': 'Saneamento - Água', 'Fórmula Aplicada': '(Poço + Nascente + Carro-pipa + Chuva + Rios + Outros) / Total Domicílios', 'Variáveis Originais (Censo 2022)': '(V00112 a V00118) / V00001'},
    {'Dimensão': 'Saneamento - Esgoto', 'Fórmula Aplicada': '(Fossa Rudimentar + Vala + Rio + Outra Forma + Inexistente) / Total Domicílios', 'Variáveis Originais (Censo 2022)': '(V00312 a V00316) / V00001'},
    {'Dimensão': 'Saneamento - Lixo', 'Fórmula Aplicada': '(Caçamba + Queimado + Enterrado + Terreno Baldio + Outros) / Total Domicílios', 'Variáveis Originais (Censo 2022)': '(V00398 a V00402) / V00001'},
    {'Dimensão': 'Educação', 'Fórmula Aplicada': 'Analfabetos (15 anos ou mais) / População Total (15 anos ou mais)', 'Variáveis Originais (Censo 2022)': 'V00901 / V00900'},
    {'Dimensão': 'Demografia (Cor/Raça)', 'Fórmula Aplicada': '(População Preta + Parda + Indígena) / População Total', 'Variáveis Originais (Censo 2022)': '(V01318 + V01320 + V01321) / v0001'},
    {'Dimensão': 'Habitação', 'Fórmula Aplicada': 'Variável pronta do IBGE que expurga domicílios coletivos automaticamente.', 'Variáveis Originais (Censo 2022)': 'v0005'},
    {'Dimensão': 'Renda', 'Fórmula Aplicada': '(Renda Máxima Região - Renda Média Setor) / (Renda Máxima - Renda Mínima)', 'Variáveis Originais (Censo 2022)': 'V06004 (Normalizada)'},
]
df_dic_app = pd.DataFrame(dados_app)

## 4. Prova Real: Validação Estatística
Calcula estatísticas (mínimo, máximo, média, nulos) para validar a base e garantir que os índices estão no padrão esperado.

In [ ]:
colunas_calculadas = ['ind_agua_inadequada', 'ind_esgoto_inadequado', 'ind_lixo_inadequado', 
                      'ind_analfabetismo', 'ind_cor_raca', 'ind_densidade_habitacional', 'ind_vulnerabilidade_renda']

df_prova = df_ivs[colunas_calculadas].describe().T[['min', 'max', 'mean']]
df_prova['valores_nulos'] = df_ivs[colunas_calculadas].isnull().sum()
df_prova = df_prova.reset_index().rename(columns={'index': 'Indicador Matemático', 'min': 'Valor Mínimo', 'max': 'Valor Máximo', 'mean': 'Média Geral'})

## 5. Exportando para Excel Bonito e Formatado
Exporta os dados para um arquivo Excel com quatro abas, aplicando formatações visuais para facilitar a análise.

In [ ]:
caminho_excel_final = caminho_bd + 'Base_IVS_Final_Formatada.xlsx'
print("Gerando o arquivo Excel com 4 abas... (Aguarde alguns minutos)")

with pd.ExcelWriter(caminho_excel_final, engine='xlsxwriter') as writer:
    # Salvando as abas
    df_ivs.to_excel(writer, sheet_name='Base_Analitica', index=False)
    df_dic_dados.to_excel(writer, sheet_name='Dicionario_Dados', index=False)
    df_dic_app.to_excel(writer, sheet_name='Dicionario_Aplicacao', index=False)
    df_prova.to_excel(writer, sheet_name='Prova_Real_Estatistica', index=False)
    
    # Formatando visualmente o Excel
    workbook = writer.book
    
    # Estilos
    formato_cabecalho = workbook.add_format({'bold': True, 'bg_color': '#1F497D', 'font_color': 'white', 'border': 1, 'align': 'center', 'valign': 'vcenter'})
    formato_numeros = workbook.add_format({'num_format': '0.0000', 'border': 1})
    formato_texto = workbook.add_format({'border': 1, 'text_wrap': True, 'valign': 'top'})
    
    # Aba 1: Base Analítica
    ws1 = writer.sheets['Base_Analitica' ]
    for col_num, value in enumerate(df_ivs.columns.values):
        ws1.write(0, col_num, value, formato_cabecalho)
    ws1.set_column('A:A', 18, formato_texto) # CD_SETOR
    ws1.set_column('B:B', 25, formato_texto) # MUN
    ws1.set_column('C:D', 20, formato_texto) # BAIRRO E SITUACAO
    ws1.set_column('E:K', 22, formato_numeros) # Índices
    ws1.freeze_panes(1, 0)
    ws1.autofilter(0, 0, len(df_ivs), len(df_ivs.columns) - 1)
    
    # Aba 2: Dicionário de Dados
    ws2 = writer.sheets['Dicionario_Dados']
    for col_num, value in enumerate(df_dic_dados.columns.values):
        ws2.write(0, col_num, value, formato_cabecalho)
    ws2.set_column('A:A', 30, formato_texto)
    ws2.set_column('B:B', 20, formato_texto)
    ws2.set_column('C:C', 80, formato_texto)
    
    # Aba 3: Dicionário de Aplicação
    ws3 = writer.sheets['Dicionario_Aplicacao']
    for col_num, value in enumerate(df_dic_app.columns.values):
        ws3.write(0, col_num, value, formato_cabecalho)
    ws3.set_column('A:A', 25, formato_texto)
    ws3.set_column('B:B', 80, formato_texto)
    ws3.set_column('C:C', 40, formato_texto)
    
    # Aba 4: Prova Real Estatística
    ws4 = writer.sheets['Prova_Real_Estatistica']
    for col_num, value in enumerate(df_prova.columns.values):
        ws4.write(0, col_num, value, formato_cabecalho)
    ws4.set_column('A:A', 30, formato_texto)
    ws4.set_column('B:D', 18, formato_numeros)
    ws4.set_column('E:E', 18, formato_texto)

print(f"SENSACIONAL! O arquivo final foi gerado perfeitamente em:\n{caminho_excel_final}")
print("Abra o arquivo no Excel e confira a aba 'Prova_Real_Estatistica' para comprovar que os cálculos deram certo!")